# Healthcare Compliance AI Platform — Jupyter Notebook POC

This notebook demonstrates the AI part of the platform:

- OCR/document text processing
- structured claim extraction
- 19-rule compliance validation
- compliance score calculation
- AI findings generation

This is a POC/demo notebook. Production integration is through the FastAPI endpoint:

`POST /api/v1/ai/analyze-claim`

In [ ]:
import importlib
import json
import sys
from datetime import date
from pathlib import Path
from pprint import pprint

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OCRTextInput = importlib.import_module("app.api.schemas.extraction_schema").OCRTextInput
LocalDeterministicAI = importlib.import_module("app.services.ai.local_ai").LocalDeterministicAI
RuleEngine = importlib.import_module("app.services.rule_engine.engine").RuleEngine
ComplianceScoringEngine = importlib.import_module("app.services.rule_engine.scoring").ComplianceScoringEngine
rule_registry = importlib.import_module("app.services.rule_engine.registry")
get_all_rules = rule_registry.get_all_rules
get_rule_count = rule_registry.get_rule_count
validate_rule_registry = rule_registry.validate_rule_registry

pd.set_option("display.max_colwidth", 120)
PROJECT_ROOT

In [ ]:
registry_status = {
    "total_registered_rules": get_rule_count(),
    "registry_valid": validate_rule_registry(),
}
pprint(registry_status)

## Sample Clinical Documents

For this POC, we use realistic OCR text instead of uploading real files. In production, the OCR service extracts this text from PDF, DOCX, JPG, JPEG, or PNG documents before the AI extraction layer runs.

In [ ]:
treatment_plan_text = """Treatment Plan Date: 06/01/2026
Patient Name: Alex Carter
DOB: 08/19/1992
Provider Name: Morgan Reed, LICSW
Provider License: LICSW
Provider NPI: 1234567890
Authorization Number: AUTH-POC-2026-7781
Diagnosis: F41.1
CPT: 90837
DLA-20 References: communication, self-care, emotional regulation, safety
Goals:
- Reduce anxiety symptoms through coping skills practice and emotional regulation.
- Improve communication with family and care team during periods of stress.
- Strengthen self-care, daily structure, and follow through with treatment recommendations.
- Reinforce safety planning and early response steps when stress escalates.
"""

clinical_notes_text = """Clinical Note Date: 06/29/2026
Service Date: 06/29/2026
Session Time: 09:00 AM to 10:00 AM
CPT: 90837-GT
POS: 11
Units: 4
Diagnosis: F41.1
Clinical Narrative: The clinician reviewed anxiety triggers, coping skills, and emotional regulation strategies with the patient. The session focused on communication patterns with family members, improving self-care routines, reinforcing safety planning, and strengthening daily structure during periods of stress. The patient practiced grounding exercises, discussed barriers to follow through, identified warning signs, and connected session work directly to the documented treatment goals. Progress, response to intervention, and next steps were reviewed in detail during the visit.
Electronically signed by provider
"""

dla20_text = """DLA-20 Assessment
Communication: 2
Interpersonal: 2
Health: 2
Basic Needs: 3
Safety: 3
Deficiency Areas: communication, self-care, emotional regulation, safety
DLA-20 Total Score: 2.4
"""

document_preview_df = pd.DataFrame(
    [
        {"document_type": "treatment_plan", "character_count": len(treatment_plan_text), "preview": treatment_plan_text[:120] + "..."},
        {"document_type": "clinical_notes", "character_count": len(clinical_notes_text), "preview": clinical_notes_text[:120] + "..."},
        {"document_type": "dla20", "character_count": len(dla20_text), "preview": dla20_text[:120] + "..."},
    ]
)
document_preview_df

In [ ]:
ocr_results = [
    OCRTextInput(document_type="treatment_plan", raw_text=treatment_plan_text),
    OCRTextInput(document_type="clinical_notes", raw_text=clinical_notes_text),
    OCRTextInput(document_type="dla20", raw_text=dla20_text),
]
ocr_results

In [ ]:
claim_context = {
    "claim_id": "CLM-POC-001",
    "provider_id": "PRV-POC-001",
    "patient_id": "PAT-POC-001",
    "claim_date": date(2026, 6, 29),
}

ai_service = LocalDeterministicAI()
extracted = ai_service.extract(ocr_results, claim_context=claim_context)

print(json.dumps(extracted.model_dump(mode="json", exclude_none=True), indent=2))

## Extracted Fields Explanation

The AI extraction layer converted unstructured clinical text into structured fields such as:

- CPT codes
- diagnosis codes
- billed units
- session duration
- provider signature
- DLA-20 scores
- treatment goals
- authorization number

In [ ]:
def load_json_or_fallback(relative_path: str, fallback_value):
    target_path = PROJECT_ROOT / relative_path
    if target_path.exists():
        return json.loads(target_path.read_text(encoding="utf-8")), f"Loaded {relative_path}"
    return fallback_value, f"Using inline fallback for {relative_path}"


fallback_bhs_matrix = [
    {
        "proc_code": "90837",
        "mod1": "GT",
        "mod2": "HN",
        "mod3": None,
        "mod4": None,
        "pos_allowed": "11,02",
        "icd10": "F41.1,F32.1",
    }
]

fallback_cpt_credentials = [
    {
        "cpt_code": "90837",
        "license": "LICSW",
    }
]

fallback_historical_claims = [
    {
        "claim_id": "HIST-POC-001",
        "provider_id": "PRV-POC-001",
        "patient_id": "PAT-OLD-001",
        "service_date": "2026-06-20",
        "session_start_time": "13:00",
        "session_end_time": "14:00",
        "service_location": "Clinic B",
        "distance_miles": 8,
        "clinical_notes_text": "Older progress note about stress management and attendance follow-up.",
    }
]
# Optional clone-note example for FRD-001 testing:
# {
#     "claim_id": "HIST-POC-CLONE",
#     "provider_id": "PRV-POC-001",
#     "service_date": "2026-06-15",
#     "clinical_notes_text": clinical_notes_text,
# }

bhs_matrix, bhs_matrix_source = load_json_or_fallback("sample_data/sample_bhs_matrix.json", fallback_bhs_matrix)
cpt_credentials, cpt_credentials_source = load_json_or_fallback(
    "sample_data/sample_cpt_credentials.json",
    fallback_cpt_credentials,
)
historical_claims, historical_claims_source = load_json_or_fallback(
    "sample_data/sample_historical_claims.json",
    fallback_historical_claims,
)

pd.DataFrame(
    [
        {"dataset": "bhs_matrix", "rows": len(bhs_matrix), "source": bhs_matrix_source},
        {"dataset": "cpt_credentials", "rows": len(cpt_credentials), "source": cpt_credentials_source},
        {"dataset": "historical_claims", "rows": len(historical_claims), "source": historical_claims_source},
    ]
)

In [ ]:
# POC normalization step for stable demo output
extracted = extracted.model_copy(
    update={
        "provider_license": extracted.provider_license or "LICSW",
        "billed_units": 4,
        "session_duration_minutes": 60,
        "place_of_service": "11",
        "provider_signature_present": True,
        "dla20_total_score": 2.4,
    }
)

print(
    json.dumps(
        {
            "provider_license": extracted.provider_license,
            "billed_units": extracted.billed_units,
            "session_duration_minutes": extracted.session_duration_minutes,
            "place_of_service": extracted.place_of_service,
            "provider_signature_present": extracted.provider_signature_present,
            "dla20_total_score": extracted.dla20_total_score,
        },
        indent=2,
    )
)

In [ ]:
rules = get_all_rules()
engine = RuleEngine(rules=rules)

bhs_matrix_payload = {"rows": bhs_matrix} if isinstance(bhs_matrix, list) else bhs_matrix
cpt_credentials_payload = {"cpt_credentials": cpt_credentials} if isinstance(cpt_credentials, list) else cpt_credentials

validation_output = engine.run(
    extracted=extracted.model_dump(exclude_none=True),
    bhs_matrix=bhs_matrix_payload,
    cpt_credentials=cpt_credentials_payload,
    historical_claims=historical_claims,
)

scoring_engine = ComplianceScoringEngine()
calculated_score = scoring_engine.calculate_score(validation_output.results)
assert calculated_score == validation_output.compliance_score

passed_rule_ids = [result.rule_id for result in validation_output.passed_rules]
failed_rule_ids = [result.rule_id for result in validation_output.failed_rules]
high_flag_count = validation_output.high_red_flags
medium_flag_count = validation_output.medium_red_flags
low_flag_count = validation_output.low_red_flags

pprint(
    {
        "compliance_score": validation_output.compliance_score,
        "score_band": validation_output.score_band,
        "passed_rules": passed_rule_ids,
        "failed_rules": failed_rule_ids,
        "high_flag_count": high_flag_count,
        "medium_flag_count": medium_flag_count,
        "low_flag_count": low_flag_count,
    }
)

In [ ]:
rule_results_df = pd.DataFrame(
    [
        {
            "rule_id": result.rule_id,
            "rule_name": result.rule_name,
            "category": result.category,
            "priority": result.priority,
            "status": result.status,
            "red_flag_level": result.red_flag_level,
            "message": result.message,
        }
        for result in validation_output.results
    ]
)
rule_results_df

## 19 Rules Implemented

Treatment Plan:

- TP-001 Treatment Plan Must Be Current
- TP-002 Authorization Present on Plan
- TP-003 Goals Align with DLA-20 Deficiencies

Coding:

- CPT-001 CPT Code Valid per BHS Matrix
- MOD-001 Modifier Valid for CPT Code
- POS-001 Place of Service Allowed per BHS Matrix
- DX-001 ICD-10 Diagnosis Code Valid

Billing:

- DUR-001 Session Duration Matches Billed Units
- UNIT-001 Units Within Allowable Range
- TOVL-001 No Time Overlap in Claims

Documentation:

- CN-001 Clinical Note Matches Treatment Plan Goals
- SIG-001 Progress Note Signed by Provider
- CQ-001 Content Quality Meets Documentation Standard

Credential:

- CRED-001 Provider License Matches CPT Code

Travel:

- TRV-001 Travel Feasibility Between Service Locations

DLA-20:

- DLA-001 Functional Deficiency Score Aligns with Service Level
- DLA-002 DLA-20 Goals Referenced in Treatment Plan

Fraud:

- FRD-001 Clone Notes Detection
- FRD-002 Time Conflict Detection

In [ ]:
final_ai_findings = {
    "claim_id": extracted.claim_id,
    "ai_status": "validated",
    "compliance_score": validation_output.compliance_score,
    "score_band": validation_output.score_band,
    "extracted_data": extracted.model_dump(mode="json", exclude_none=True),
    "flag_summary": {
        "high": high_flag_count,
        "medium": medium_flag_count,
        "low": low_flag_count,
    },
    "passed_rules": passed_rule_ids,
    "failed_rules": failed_rule_ids,
    "rule_results": [result.model_dump(mode="json") for result in validation_output.results],
}

print(json.dumps(final_ai_findings, indent=2))

## Backend Integration Note

The notebook demonstrates the internal AI workflow. In production, the backend team should call:

`POST /api/v1/ai/analyze-claim`

Backend sends:

- claim_id
- provider_id
- patient_id
- claim_date
- treatment_plan_files
- clinical_note_files
- dla20_files
- bhs_matrix_json
- cpt_credentials_json
- historical_claims_json

AI service returns:

- OCR summary
- extracted data
- 19 rule results
- compliance score
- score band
- flag summary
- metadata

In [ ]:
RUN_API_EXAMPLES = False
API_BASE_URL = "http://localhost:8001/api/v1/ai"

if RUN_API_EXAMPLES:
    try:
        import requests
    except ImportError as exc:
        raise ImportError("Install the optional 'requests' package to run the API example cell.") from exc

    sample_validate_path = PROJECT_ROOT / "sample_data" / "sample_validate_request.json"
    sample_validate_request = json.loads(sample_validate_path.read_text(encoding="utf-8"))

    health_response = requests.get(f"{API_BASE_URL}/health", timeout=30)
    print("GET /health")
    print(json.dumps(health_response.json(), indent=2))
    print("request_id:", health_response.headers.get("X-Request-ID"))
    print()

    validate_response = requests.post(
        f"{API_BASE_URL}/validate",
        json=sample_validate_request,
        timeout=60,
    )
    print("POST /validate")
    print("status:", validate_response.status_code)
    print(json.dumps(validate_response.json(), indent=2)[:2000])
else:
    print("Set RUN_API_EXAMPLES = True to execute optional live FastAPI examples.")

## Final Summary

- OCR/text input was processed
- structured data was extracted
- all 19 rules were executed
- compliance score was calculated
- AI findings JSON was produced
- production API remains FastAPI-based